# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### In Your Response:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load dataset
url = 'https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/refs/heads/main/DataSets/hotels.csv'
hotels = pd.read_csv(url)

# Basic shape check
print("Dataset shape:", hotels.shape)
print("\nColumns:\n", hotels.columns.tolist())

# Drop leakage columns that reveal the outcome
# These would make the model unrealistically good
drop_cols = ['reservation_status', 'reservation_status_date']
hotels = hotels.drop(columns=drop_cols, errors='ignore')

# Separate features and target
X = hotels.drop(columns=['is_canceled'])
y = hotels['is_canceled']

# Identify numeric and categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(exclude=['object']).columns.tolist()

print("\nCategorical columns:")
print(categorical_cols)

print("\nNumeric columns:")
print(numeric_cols)

# Split BEFORE preprocessing
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

print("\nTraining shape:", X_train_raw.shape)
print("Test shape:", X_test_raw.shape)

# Missing values check
print("\nMissing values:")
print(hotels.isnull().sum().sort_values(ascending=False).head(15))

Dataset shape: (9404, 32)

Columns:
 ['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']

Categorical columns:
['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type']

Numeric columns:
['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_

### ✍️ Your Response: 🔧
The dataset contains 9,404 rows and 32 columns originally. After removing the leakage columns reservation_status and reservation_status_date, the modeling dataset includes 29 features plus the target variable. The dataset includes both categorical features, such as hotel, arrival_date_month, meal, country, market_segment, distribution_channel, reserved_room_type, assigned_room_type, and deposit_type, as well as numerical features like lead_time, adr, previous_cancellations, and total_of_special_requests.

To prepare the data, I removed columns that would leak the outcome, separated the features (X) and target variable (is_canceled), and split the dataset into training and test sets before any preprocessing. I also identified categorical and numerical columns and examined missing values, which were later handled using imputation in the modeling pipelines.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import time

# Preprocessing for Naive Bayes
# MultinomialNB requires nonnegative values, so use MinMaxScaler
nb_preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', MinMaxScaler())
        ]), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_cols)])

start_time = time.time()

X_train_nb = nb_preprocessor.fit_transform(X_train_raw)
X_test_nb = nb_preprocessor.transform(X_test_raw)

nb_model = MultinomialNB()
nb_model.fit(X_train_nb, y_train)

nb_train_time = time.time() - start_time

y_pred_nb = nb_model.predict(X_test_nb)
nb_accuracy = accuracy_score(y_test, y_pred_nb)

print("Naive Bayes Accuracy:", nb_accuracy)
print("Naive Bayes Training Time:", nb_train_time)

print("\nNaive Bayes Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

print("\nNaive Bayes Classification Report:")
print(classification_report(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.8423104181431609
Naive Bayes Training Time: 0.0776829719543457

Naive Bayes Confusion Matrix:
[[1984  137]
 [ 308  393]]

Naive Bayes Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.94      0.90      2121
           1       0.74      0.56      0.64       701

    accuracy                           0.84      2822
   macro avg       0.80      0.75      0.77      2822
weighted avg       0.83      0.84      0.83      2822



### ✍️ Your Response: 🔧
The Naïve Bayes model performed reasonably well as a baseline, achieving an accuracy of about 0.84. However, accuracy alone is not the best metric for this problem because the dataset is imbalanced, with more non-canceled bookings than canceled ones. A more important metric is recall for the canceled class, which was 0.56. This indicates that the model is missing a significant portion of actual cancellations, which could be costly for the business.

This model could still be useful for the hotel in situations where speed and simplicity are important, such as real-time alerts or initial screening of booking risk. It provides a quick and efficient way to flag potentially risky bookings, but it may need to be supplemented with a more advanced model to improve detection of cancellations.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import time

# Preprocessing for SVM
svm_preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_cols)])

start_time = time.time()

X_train_svm = svm_preprocessor.fit_transform(X_train_raw)
X_test_svm = svm_preprocessor.transform(X_test_raw)

svm_model = LinearSVC(random_state=42, max_iter=10000)
svm_model.fit(X_train_svm, y_train)

svm_train_time = time.time() - start_time

y_pred_svm = svm_model.predict(X_test_svm)
svm_accuracy = accuracy_score(y_test, y_pred_svm)

print("SVM Accuracy:", svm_accuracy)
print("SVM Training Time:", svm_train_time)

print("\nSVM Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

print("\nSVM Classification Report:")
print(classification_report(y_test, y_pred_svm))

SVM Accuracy: 0.9011339475549256
SVM Training Time: 0.14643526077270508

SVM Confusion Matrix:
[[2019  102]
 [ 177  524]]

SVM Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.95      0.94      2121
           1       0.84      0.75      0.79       701

    accuracy                           0.90      2822
   macro avg       0.88      0.85      0.86      2822
weighted avg       0.90      0.90      0.90      2822



### ✍️ Your Response: 🔧
1. The SVM model performed very well, achieving an accuracy of about 0.90. In addition to accuracy, recall and F1-score for the canceled class are especially important because the business wants to correctly identify bookings that are likely to cancel. The recall for canceled bookings improved to 0.75, which is significantly better than the Naïve Bayes model and shows that the SVM is more effective at detecting cancellations.

2. SVM can provide better insights than simpler models when customer behavior depends on more complex patterns across multiple features, such as lead time, deposit type, and previous booking behavior. It is especially useful when relationships in the data are not purely linear or obvious, allowing the hotel to make more informed decisions about cancellation risk.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

### In Your Response:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import time

# True validation split from training data only
X_train_nn_raw, X_val_nn_raw, y_train_nn, y_val_nn = train_test_split(
    X_train_raw, y_train, test_size=0.2, random_state=42, stratify=y_train)

# Preprocessing for neural network
nn_preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_cols)])

start_time = time.time()

X_train_nn = nn_preprocessor.fit_transform(X_train_nn_raw)
X_val_nn = nn_preprocessor.transform(X_val_nn_raw)
X_test_nn = nn_preprocessor.transform(X_test_raw)

# Simple 2 hidden layer model
nn_model = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    random_state=42,
    max_iter=300)

nn_model.fit(X_train_nn, y_train_nn)

nn_train_time = time.time() - start_time

y_val_pred_nn = nn_model.predict(X_val_nn)
y_pred_nn = nn_model.predict(X_test_nn)

val_accuracy_nn = accuracy_score(y_val_nn, y_val_pred_nn)
nn_accuracy = accuracy_score(y_test, y_pred_nn)

print("Neural Network Validation Accuracy:", val_accuracy_nn)
print("Neural Network Test Accuracy:", nn_accuracy)
print("Neural Network Training Time:", nn_train_time)

print("\nNeural Network Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_nn))

print("\nNeural Network Classification Report:")
print(classification_report(y_test, y_pred_nn))

Neural Network Validation Accuracy: 0.932422171602126
Neural Network Test Accuracy: 0.9107016300496102
Neural Network Training Time: 6.113652467727661

Neural Network Confusion Matrix:
[[2011  110]
 [ 142  559]]

Neural Network Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.95      0.94      2121
           1       0.84      0.80      0.82       701

    accuracy                           0.91      2822
   macro avg       0.88      0.87      0.88      2822
weighted avg       0.91      0.91      0.91      2822



/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


### ✍️ Your Response: 🔧
The neural network was flexible and powerful, and it performed competitively compared with the other models. However, it was harder to interpret and usually took longer to train.
A business may hesitate to rely fully on a black box model like this because managers often want to understand why a booking was classified as high risk. Even if the model performs well, its lower interpretability can make it harder to explain, trust, and use in day-to-day operational decisions.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

### In Your Response:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?


In [11]:
comparison = pd.DataFrame({
    'Model': ['Naive Bayes', 'SVM', 'Neural Network'],
    'Accuracy': [nb_accuracy, svm_accuracy, nn_accuracy],
    'Training Time (seconds)': [nb_train_time, svm_train_time, nn_train_time]})

comparison = comparison.sort_values(by='Accuracy', ascending=False)

print(comparison)

            Model  Accuracy  Training Time (seconds)
2  Neural Network  0.910702                 6.113652
1             SVM  0.901134                 0.146435
0     Naive Bayes  0.842310                 0.077683


### ✍️ Your Response: 🔧
1. The Neural Network had the best overall accuracy at approximately 0.91, followed closely by the SVM at about 0.90, while Naïve Bayes performed the weakest at around 0.84. In terms of training time, Naïve Bayes and SVM were very fast, while the Neural Network took significantly longer to train. For interpretability, Naïve Bayes was the easiest to understand, SVM was moderately interpretable, and the Neural Network was the most complex and least interpretable.

2. I would recommend the SVM model for deployment because it provides nearly the same high accuracy as the Neural Network while being much faster and easier to interpret. Although the Neural Network had slightly higher accuracy, its longer training time and “black box” nature make it less practical for business use. The SVM offers a strong balance between performance, efficiency, and usability for the hotel’s decision-making needs.

## 6. Final Business Recommendation

### In Your Response:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?
2. How does this relate to your customized learning outcome you created in canvas?


1. I recommend implementing the SVM model to predict booking cancellations. Although the Neural Network achieved the highest accuracy (~0.91), the SVM performed nearly as well (~0.90) while being faster, more stable, and easier to interpret. This makes it more practical for real business use, where managers need reliable and understandable insights. The model helps address the problem of unpredictable cancellations by identifying high-risk bookings in advance, allowing the hotel to better manage staffing, room inventory, and revenue planning.

2. A limitation is that customer behavior may change over time, which could reduce model accuracy if not updated regularly. Additional data such as customer loyalty history, booking timing patterns, and payment behavior could improve predictions. This aligns with my learning outcome of applying machine learning models to support real-world business decisions.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [12]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"

[NbConvertApp] WARNING | pattern 'assignment_12_bayes_svm_neural.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=